# QA Evaluation Set -- Final Combined Notebook

This notebook covers the full pipeline for building the 138 question and answer evaluation set based on the collected Caribbean exchange data.

**Note: A free HuggingFace token can be acquired from the website: https://huggingface.co/. 

## Generate Question Candidates

In [1]:
# Import the required packages
import json
import random
from collections import defaultdict
from pathlib import Path
import pandas as pd

# Load in the exchange corpus
with open("unified_text_corpus.json", "r", encoding="utf-8") as f:
    corpus = json.load(f)

# Print the number of loaded documents
print(f"Loaded {len(corpus)} documents.")

# Save the corpus in a DataFrame
df = pd.DataFrame(corpus)

# Print the number of documents by exchange
print(df.groupby(["exchange", "doc_type"]).size())

Loaded 9603 documents.
exchange  doc_type                   
bse       annual_report_or_financials     133
          disclosure                      106
jse       annual_report_or_financials    3244
          disclosure                     4972
ttse      general_document               1148
dtype: int64


In [2]:
# Set the random seed value
random.seed(42)

# Set the targer number of questions for each exchange and question category
TARGET_COUNTS = {
    ("jse", "factual"): 17,
    ("jse", "event"): 17,
    ("jse", "comparative"): 12,
    ("bse", "factual"): 17,
    ("bse", "event"): 17,
    ("bse", "comparative"): 12,
    ("ttse", "factual"): 17,
    ("ttse", "event"): 17,
    ("ttse", "comparative"): 12,}

# Print the total number of questions
print(f"Total target questions: {sum(TARGET_COUNTS.values())}")

# Print the total number of questions by exchange
print("\nBy exchange:")
by_exchange = defaultdict(int)
for (exch, cat), n in TARGET_COUNTS.items():
    by_exchange[exch] += n
for exch, n in by_exchange.items():
    print(f"  {exch}: {n}")

Total target questions: 138

By exchange:
  jse: 46
  bse: 46
  ttse: 46


### Generate Comparative Question Candidates

In [3]:
# Import the required packages
import re
from datetime import datetime
from collections import defaultdict
from dateutil import parser as date_parser

# Test to confirm dateutil is successful
try:
    from dateutil import parser as date_parser
    _test = date_parser.parse("2026-06-15T09:23:00", fuzzy=True)
    print("dateutil works:", _test)
    DATEUTIL_AVAILABLE = True
except Exception as e:
    print("dateutil FAILED:", e, "-- falling back to manual format list.")
    DATEUTIL_AVAILABLE = False

# Print a sample a few title/date pairs
print("\nSample JSE title/date pairs:")
for r in [r for r in corpus if r["exchange"] == "jse" and r.get("date")][:5]:
    print(" ", repr(r.get("title")), "|", repr(r.get("date")))

# Define a function to extract the ticker symbol from each document
def extract_ticker(title: str):
    matches = re.findall(r"\(([A-Z]{2,10}(?:\.[A-Z0-9]+)?)\)", title or "")
    return matches[0] if matches else None

# Define a function to extract the company key from the document
def extract_company_key(record: dict) -> str:
   
    title = record.get("title", "") or ""

    # JSE ticker style
    ticker_match = re.findall(r"\(([A-Z]{2,10}(?:\.[A-Z0-9]+)?)\)", title)
    if ticker_match:
        return ticker_match[0]

    # BSE ticker style
    parts = re.split(r"\s[–—-]\s", title)
    if len(parts) >= 2:
        candidate_name = parts[1].strip()
        if candidate_name and not re.match(r"^\d", candidate_name):
            return candidate_name.lower()

    return (record.get("company") or "").strip().lower() or None

# Set the date expected formats
MANUAL_DATE_FORMATS = ["%Y-%m-%dT%H:%M:%S", "%Y-%m-%d", "%d-%m-%Y", "%B %d, %Y", "%d %B %Y", "%Y"]

# Define a function to extract the date
def try_parse_date(date_str):
    if not date_str:
        return None
    date_str = str(date_str).strip()

    if DATEUTIL_AVAILABLE:
        try:
            return date_parser.parse(date_str, fuzzy=True)
        except (ValueError, OverflowError):
            pass

    for fmt in MANUAL_DATE_FORMATS:
        try:
            return datetime.strptime(date_str[:len(fmt) + 6], fmt)
        except ValueError:
            continue
    return None

# Define a function to find comparative pair documents
def find_comparative_pairs_text(records: list) -> list:
    by_company = defaultdict(list)
    unmatched_dates = 0
    for r in records:
        company_key = extract_company_key(r)
        if not company_key:
            continue
        parsed_date = try_parse_date(r.get("date"))
        if parsed_date is None:
            unmatched_dates += 1
            continue
        key = (r["exchange"], company_key)
        by_company[key].append((parsed_date, r))

    print(f"  (text pairing: {unmatched_dates} records had an unparseable date)")

    pairs = []
    for key, entries in by_company.items():
        if len(entries) < 2:
            continue
        entries.sort(key=lambda x: x[0])
        pairs.append((entries[0][1], entries[-1][1]))
    return pairs

# Generate potential comparative pair candidates
text_comparative_pairs = find_comparative_pairs_text(corpus)
print(f"\nText-based: found {len(text_comparative_pairs)} companies with multiple dated records.")

# Print the results by exchange
by_exchange_text = defaultdict(int)
for a, b in text_comparative_pairs:
    by_exchange_text[a["exchange"]] += 1
print(dict(by_exchange_text))

dateutil works: 2026-06-15 09:23:00

Sample JSE title/date pairs:
  'Tropical Battery Company Limited (TROPICAL) – Interim Financial Statements For The Third Quarter Ended June 30, 2026' | '2026-08-14T19:43:27'
  'PROVEN GROUP LIMITED UNAUDITED FIRST QUARTER RESULTS JUNE 30 2026 AND UNAUDITED FINANCIAL YEAR 2026 PERFORMANCE' | '2026-08-14T19:43:05'
  'Dolphin Cove Limited (DCOVE) – Unaudited Financial Statements for the Second Quarter Ended 30 June 2026.' | '2026-08-14T19:36:32'
  'Kingston Properties Limited (KPREIT) Unaudited Financial Statements For The Second Quarter Ended June 30, 2026' | '2026-08-14T19:31:17'
  'Eppley Caribbean Property Fund Limited SCC (CPFV) – Value Fund Net Asset Value (NAV) August 14, 2026' | '2026-08-14T19:29:50'
  (text pairing: 26 records had an unparseable date)

Text-based: found 179 companies with multiple dated records.
{'jse': 150, 'bse': 14, 'ttse': 15}


In [4]:
# Define key words to search for comparative candidates
DIVIDEND_KEYWORDS = ["dividend"]
PROFIT_KEYWORDS = ["annual report", "audited financial statements", "annual results",
                   "unaudited financial statements", "financial statements"]

# Define a function to find comparative pairs with the listed key words
def find_comparative_pairs_by_topic(records: list, keywords: list) -> list:
    
    filtered = [r for r in records if any(k in (r.get("title", "") or "").lower() for k in keywords)]
    return find_comparative_pairs_text(filtered)

# Find dividend and profit candidate pairs
dividend_pairs = find_comparative_pairs_by_topic(corpus, DIVIDEND_KEYWORDS)
profit_pairs = find_comparative_pairs_by_topic(corpus, PROFIT_KEYWORDS)

# Print the number of pairs found
print(f"Dividend comparison pairs: {len(dividend_pairs)}")
print(f"Profit comparison pairs: {len(profit_pairs)}")

  (text pairing: 2 records had an unparseable date)
  (text pairing: 0 records had an unparseable date)
Dividend comparison pairs: 76
Profit comparison pairs: 116


In [5]:
# Define a function to find comparative pairs by price
def find_price_pairs(csv_path: Path, exchange: str, company_col_candidates, date_col_candidates, value_col_candidates):

    if not csv_path.exists():
        print(f"  {csv_path} not found -- skipping.")
        return []

    sdf = pd.read_csv(csv_path)
    sdf.columns = [c.lower() for c in sdf.columns]

    def find_col(candidates):
        for cand in candidates:
            for col in sdf.columns:
                if cand in col:
                    return col
        return None

    company_col = find_col(company_col_candidates)
    date_col = find_col(date_col_candidates)
    value_col = find_col(value_col_candidates)

    if not (company_col and date_col and value_col):
        print(f"  {csv_path.name}: could not find matching columns "
              f"(company={company_col}, date={date_col}, value={value_col}) -- skipping.")
        return []

    by_company = defaultdict(list)
    for _, row in sdf.iterrows():
        company = str(row[company_col]).strip()
        parsed_date = try_parse_date(row[date_col])
        value = row[value_col]
        if not company or company.lower() == "nan" or parsed_date is None or pd.isna(value):
            continue
        by_company[company].append((parsed_date, row[date_col], value))

    pairs = []
    for company, entries in by_company.items():
        if len(entries) < 2:
            continue
        entries.sort(key=lambda x: x[0])
        earliest, latest = entries[0], entries[-1]
        pairs.append({
            "exchange": exchange,
            "company": company,
            "date_earlier": str(earliest[1]),
            "value_earlier": earliest[2],
            "date_later": str(latest[1]),
            "value_later": latest[2],
            "value_col": value_col,
            "source_file": csv_path.name,
        })
    return pairs

# Create a variable to save the price pairs
price_pairs = []

# Generate price candidate pairs for each exchange
price_pairs += find_price_pairs(
    Path("cleaned_jse_price_data.csv"), "jse",
    company_col_candidates=["symbol"], date_col_candidates=["date"], value_col_candidates=["closing_price", "close"],)

price_pairs += find_price_pairs(
    Path("cleaned_bse_price_data.csv"), "bse",
    company_col_candidates=["company_name", "symbol"], date_col_candidates=["date"], value_col_candidates=["closing_price", "close"],)

price_pairs += find_price_pairs(
    Path("cleaned_ttse_daily_quotes_security_detail.csv"), "ttse",
    company_col_candidates=["security"], date_col_candidates=["last_sale_date", "date"], value_col_candidates=["close_price", "close"],)

# Print the number of price candidates generated
print(f"\nStructured-data based: found {len(structured_comparative_pairs)} companies with multiple dated records.")
by_exchange_structured = defaultdict(int)
for p in structured_comparative_pairs:
    by_exchange_structured[p["exchange"]] += 1
print(dict(by_exchange_structured))


Structured-data based: found 298 companies with multiple dated records.
{'jse': 214, 'bse': 31, 'ttse': 53}


### Generate the Factual and Event Candidates

In [6]:
# Define the function to truncate financial document texts
def truncate(text, max_chars = 3500):
    return text[:max_chars] + ("..." if len(text) > max_chars else "")

# Create a variable to save the candidates
candidates = []

# For each exchange generate factual and event candidates
for exchange in ["jse", "bse", "ttse"]:
    exchange_docs = [r for r in corpus if r["exchange"] == exchange and r["text"]]
    random.shuffle(exchange_docs)

    for category in ["factual", "event"]:
        target = TARGET_COUNTS.get((exchange, category), 0)
        selected = exchange_docs[:target]
        exchange_docs = exchange_docs[target:]

        for doc in selected:
            candidates.append({
                "candidate_id": f"{exchange}-{category}-{len(candidates)}",
                "exchange": exchange, "category": category,
                "company": doc.get("company"), "date": doc.get("date"),
                "doc_type": doc.get("doc_type"), "source_doc_id": doc.get("id"),
                "source_url": doc.get("url"), "text_excerpt": truncate(doc["text"]),
            })

# Group each comparative pair-type by exchange, so sampling is done by exchange
dividend_pairs_by_exchange = defaultdict(list)
for a, b in dividend_pairs:
    dividend_pairs_by_exchange[a["exchange"]].append((a, b))

profit_pairs_by_exchange = defaultdict(list)
for a, b in profit_pairs:
    profit_pairs_by_exchange[a["exchange"]].append((a, b))

structured_pairs_by_exchange = defaultdict(list)
for p in structured_comparative_pairs:
    structured_pairs_by_exchange[p["exchange"]].append(p)

# For each exchange split the comparative quota evenly across the three sub-types
for exchange in ["jse", "bse", "ttse"]:
    target = TARGET_COUNTS.get((exchange, "comparative"), 0)

    dividend_pool = dividend_pairs_by_exchange.get(exchange, [])
    profit_pool = profit_pairs_by_exchange.get(exchange, [])
    price_pool = structured_pairs_by_exchange.get(exchange, [])
    random.shuffle(dividend_pool)
    random.shuffle(profit_pool)
    random.shuffle(price_pool)

    even_share = target // 3
    remainder = target - even_share * 3

    dividend_take = min(even_share, len(dividend_pool))
    profit_take = min(even_share, len(profit_pool))
    shortfall = (even_share - dividend_take) + (even_share - profit_take) + remainder
    price_take = min(even_share + shortfall, len(price_pool))

    # Build candidates from the dividend candidate pool
    for a, b in dividend_pool[:dividend_take]:
        candidates.append({
            "candidate_id": f"{exchange}-comparative-{len(candidates)}",
            "exchange": exchange, "category": "comparative", "comparative_type": "dividend",
            "company": a.get("company") or a.get("title"),
            "date_earlier": a.get("date"), "date_later": b.get("date"),
            "doc_type": a.get("doc_type"),
            "source_doc_id_earlier": a.get("id"), "source_doc_id_later": b.get("id"),
            "text_excerpt_earlier": truncate(a["text"]), "text_excerpt_later": truncate(b["text"]),
        })

     # Build candidates from the profit candidate pool
    for a, b in profit_pool[:profit_take]:
        candidates.append({
            "candidate_id": f"{exchange}-comparative-{len(candidates)}",
            "exchange": exchange, "category": "comparative", "comparative_type": "profit",
            "company": a.get("company") or a.get("title"),
            "date_earlier": a.get("date"), "date_later": b.get("date"),
            "doc_type": a.get("doc_type"),
            "source_doc_id_earlier": a.get("id"), "source_doc_id_later": b.get("id"),
            "text_excerpt_earlier": truncate(a["text"]), "text_excerpt_later": truncate(b["text"]),
        })

    # Build candidates from the price candidate pool
    for p in price_pool[:price_take]:
        candidates.append({
            "candidate_id": f"{exchange}-comparative-{len(candidates)}",
            "exchange": exchange, "category": "comparative", "comparative_type": "price",
            "company": p["company"],
            "date_earlier": p["date_earlier"], "date_later": p["date_later"],
            "doc_type": "structured_price_data",
            "value_col": p["value_col"], "value_earlier": p["value_earlier"], "value_later": p["value_later"],
            "source_file": p["source_file"],
            "text_excerpt_earlier": f"On {p['date_earlier']}, {p['company']}'s {p['value_col']} was {p['value_earlier']}.",
            "text_excerpt_later": f"On {p['date_later']}, {p['company']}'s {p['value_col']} was {p['value_later']}.",
        })

    # Total the number of comparative candidates
    actual_total = dividend_take + profit_take + price_take

# Print the total number of candidates generated
print(f"Total candidates sampled: {len(candidates)}")
cand_df = pd.DataFrame(candidates)
print(cand_df.groupby(["exchange", "category"]).size())

# Print the number of comparative candidates by type
if "comparative_type" in cand_df.columns:
    print("\nComparative candidates by type:")
    print(cand_df[cand_df["category"] == "comparative"].groupby(["exchange", "comparative_type"]).size())

Total candidates sampled: 138
exchange  category   
bse       comparative    12
          event          17
          factual        17
jse       comparative    12
          event          17
          factual        17
ttse      comparative    12
          event          17
          factual        17
dtype: int64

Comparative candidates by type:
exchange  comparative_type
bse       dividend            4
          price               8
jse       dividend            4
          price               4
          profit              4
ttse      dividend            4
          price               4
          profit              4
dtype: int64


In [9]:
# Save the candidate questions for review
out_path = Path("qa_candidates_for_review.json")
out_path.write_text(json.dumps(candidates, indent = 2, ensure_ascii = False), encoding = "utf-8")
print(f"Saved {len(candidates)} candidates to {out_path.resolve()}")

Saved 138 candidates to /home/areas001/Final Project/Final Notebooks/qa_candidates_for_review.json


## Generate Questions and Answers

In [10]:
# Set the Hugging Face token
HF_TOKEN = ""

In [11]:
# Import the required package
from huggingface_hub import hf_hub_download

# Specifies the Llama build and 4-bit quantization
GGUF_REPO = "bartowski/Meta-Llama-3.1-8B-Instruct-GGUF"
GGUF_FILENAME = "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"

# Download the model file
model_path = hf_hub_download(repo_id = GGUF_REPO, filename = GGUF_FILENAME, token = HF_TOKEN)

In [12]:
# Import the required package
from llama_cpp import Llama

# Load the Llama 3.1 model
llm = Llama(
    model_path = model_path,
    n_ctx = 4096,      
    n_threads = 8,     
    verbose = False,)

In [13]:
# Load in the candidate pairs
with open("qa_candidates_for_review.json", "r", encoding="utf-8") as f:
    candidates = json.load(f)

# Print the number of candidates by category
print(f"Loaded {len(candidates)} candidates.")
by_category = {}
for c in candidates:
    by_category[c["category"]] = by_category.get(c["category"], 0) + 1
print(by_category)

Loaded 138 candidates.
{'factual': 51, 'event': 51, 'comparative': 36}


In [13]:
# Create the prompt templates for the questions and answer pairs
FACTUAL_PROMPT = """You are creating a factual question-answer pair for a financial QA evaluation dataset, based ONLY on the text below, which concerns {company} or company symbol.

Text:
{excerpt}

The question MUST explicitly mention the company name ("{company}") or company symbol so that it can be understood on its own, without needing any other context. For example, write "What was the date of {company}'s audited financial statements?" rather than "What was the date of the audited financials?"

Respond ONLY in this exact JSON format, nothing else:
{{"question": "...", "answer": "...", "answerable": true}}

If the text does not contain a clear enough fact to build a good question, respond with:
{{"question": "", "answer": "", "answerable": false}}
"""

EVENT_PROMPT = """You are creating a question-answer pair about a specific dated event (e.g. a dividend declaration, AGM or director change) concerning {company}, based ONLY on the text below.

Text:
{excerpt}

The question MUST explicitly mention the company name ("{company}") or company symbol so that it can be understood on its own, without needing any other context.

Respond ONLY in this exact JSON format, nothing else:
{{"question": "...", "answer": "...", "answerable": true}}

If the text does not describe a clear dated event, respond with:
{{"question": "", "answer": "", "answerable": false}}
"""

COMPARATIVE_PROMPT = """You are creating a comparative question-answer pair concerning {company}, based ONLY on the two excerpts below, which describe {company} at two different dates.

Excerpt from {date_earlier}:
{excerpt_earlier}

Excerpt from {date_later}:
{excerpt_later}

The question MUST explicitly mention the company name ("{company}") or company symbol so that it can be understood on its own, without needing any other context. The question should ask the reader to compare a specific fact between the two dates (e.g. how a dividend, figure, or status changed).

Respond ONLY in this exact JSON format, nothing else:
{{"question": "...", "answer": "...", "answerable": true}}

If the two excerpts do not support a clear comparison, respond with:
{{"question": "", "answer": "", "answerable": false}}
"""

In [15]:
# Define a function to extract the company name from documents
def extract_company_name(candidate: dict) -> str:
   
    if candidate.get("company"):
        return candidate["company"]

    title = candidate.get("title", "") or ""
 
    match = re.match(r"^(.*?)\s*\(", title)
    if match and match.group(1).strip():
        return match.group(1).strip()

    ticker_match = re.search(r"\(([A-Z]{2,10}(?:\.[A-Z0-9]+)?)\)", title)
    if ticker_match:
        return ticker_match.group(1)

    return "the company" 

# Define a function to build the model prompts
def build_prompt(candidate: dict) -> str:
    company = extract_company_name(candidate)
    if candidate["category"] == "factual":
        return FACTUAL_PROMPT.format(company=company, excerpt=candidate["text_excerpt"])
    elif candidate["category"] == "event":
        return EVENT_PROMPT.format(company=company, excerpt=candidate["text_excerpt"])
    elif candidate["category"] == "comparative":
        return COMPARATIVE_PROMPT.format(
            company=company,
            date_earlier=candidate.get("date_earlier", "unknown"),
            excerpt_earlier=candidate["text_excerpt_earlier"],
            date_later=candidate.get("date_later", "unknown"),
            excerpt_later=candidate["text_excerpt_later"],
        )
    raise ValueError(f"Unknown category: {candidate['category']}")

# Define a function to extract the json text
def extract_json(text: str) -> dict:
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None

# Define a function to generate the question and answer pairs
def generate_qa_pair(candidate: dict, max_tokens: int = 300) -> dict:
    prompt = build_prompt(candidate)
    messages = [
        {"role": "system", "content": "You output only valid JSON, no other text."},
        {"role": "user", "content": prompt},]

    for attempt in range(2):
        response = llm.create_chat_completion(messages=messages, max_tokens=max_tokens, temperature=0.3)
        raw_text = response["choices"][0]["message"]["content"]
        parsed = extract_json(raw_text)
        if parsed is not None:
            return parsed
        if attempt == 0:
            messages.append({"role": "user", "content": "That was not valid JSON. Respond with ONLY the JSON object, nothing else."})

    return {"question": "", "answer": "", "answerable": False, "raw_output": raw_text}

In [17]:
# Create a variable to save the QA pairs
draft_qa_pairs = []

# Set the progress path for the QA generation
progress_path = Path("qa_generation_progress.json")

# If the progress file exists, continue from the previous run
if progress_path.exists():
    with open(progress_path, "r", encoding = "utf-8") as f:
        draft_qa_pairs = json.load(f)
    processed_ids = {d["candidate_id"] for d in draft_qa_pairs}
    print(f"Resuming -- {len(draft_qa_pairs)} already generated.")
else:
    processed_ids = set()

# Print the number of remaining candidates
remaining = [c for c in candidates if c["candidate_id"] not in processed_ids]
print(f"Generating {len(remaining)} remaining QA pairs (of {len(candidates)} total)...")

# For every remaining candidate, generate and save a question, answer and the source document
for i, candidate in enumerate(remaining):
    print(f"[{i+1}/{len(remaining)}] {candidate['candidate_id']} ({candidate['category']})...", end=" ")

    result = generate_qa_pair(candidate)

    draft_qa_pairs.append({
        "candidate_id": candidate["candidate_id"],
        "exchange": candidate["exchange"],
        "category": candidate["category"],
        "question": result.get("question", ""),
        "answer": result.get("answer", ""),
        "answerable": result.get("answerable", False),
        "source_doc_id": candidate.get("source_doc_id") or
                          f"{candidate.get('source_doc_id_earlier','')} + {candidate.get('source_doc_id_later','')}",
        "source_url": candidate.get("source_url", ""),
        "generation_method": "llama-3.1-8b-instruct-q4",
        "human_verified": False,
    })

    print("OK" if result.get("answerable") else "NOT ANSWERABLE")

    if (i + 1) % 5 == 0:
        with open(progress_path, "w", encoding="utf-8") as f:
            json.dump(draft_qa_pairs, f, indent=2, ensure_ascii=False)
        print(f"  -- checkpoint saved ({len(draft_qa_pairs)} total)")

# Save the generated QAs
with open(progress_path, "w", encoding = "utf-8") as f:
    json.dump(draft_qa_pairs, f, indent = 2, ensure_ascii = False)

# Print the progress of the QA generation
print(f"\nDone. {len(draft_qa_pairs)} draft QA pairs generated.")
answerable_count = sum(1 for d in draft_qa_pairs if d["answerable"])
print(f"Answerable: {answerable_count} / {len(draft_qa_pairs)}")

Resuming -- 120 already generated.
Generating 78 remaining QA pairs (of 138 total)...
[1/78] jse-factual-15 (factual)... OK
[2/78] jse-factual-16 (factual)... OK
[3/78] jse-event-30 (event)... OK
[4/78] jse-event-31 (event)... OK
[5/78] jse-event-32 (event)... OK
  -- checkpoint saved (125 total)
[6/78] jse-event-33 (event)... OK
[7/78] bse-factual-45 (factual)... OK
[8/78] bse-factual-46 (factual)... OK
[9/78] bse-factual-47 (factual)... OK
[10/78] bse-factual-48 (factual)... OK
  -- checkpoint saved (130 total)
[11/78] bse-factual-49 (factual)... OK
[12/78] bse-factual-50 (factual)... OK
[13/78] bse-event-60 (event)... NOT ANSWERABLE
[14/78] bse-event-61 (event)... OK
[15/78] bse-event-62 (event)... OK
  -- checkpoint saved (135 total)
[16/78] bse-event-63 (event)... OK
[17/78] bse-event-64 (event)... OK
[18/78] bse-event-65 (event)... OK
[19/78] bse-event-66 (event)... OK
[20/78] bse-event-67 (event)... OK
  -- checkpoint saved (140 total)
[21/78] ttse-factual-75 (factual)... NOT AN

In [19]:
# Filter the answerable pairs
answerable_pairs = [d for d in draft_qa_pairs if d["answerable"] and d["question"] and d["answer"]]

# Save the answerable QAs to a file
out_path = Path("qa_draft_set.json")
out_path.write_text(json.dumps(answerable_pairs, indent = 2, ensure_ascii = False), encoding = "utf-8")
print(f"Saved {len(answerable_pairs)} draft QA pairs to {out_path.resolve()}")

# Save the pairs in a DataFrame for review
draft_df = pd.DataFrame(answerable_pairs)

# Print the number of pairs by exchange and category
print("\nBy exchange + category:")
print(draft_df.groupby(["exchange", "category"]).size())

Saved 192 draft QA pairs to /home/areas001/Final Project/Final Notebooks/qa_draft_set.json

By exchange + category:
exchange  category   
bse       comparative    22
          event          22
          factual        21
jse       comparative    22
          event          17
          factual        17
ttse      comparative    21
          event          27
          factual        23
dtype: int64


## QA Human Verification

In [16]:
# Load the QA draft set for review
with open("qa_draft_set.json", "r", encoding = "utf-8") as f:
    answerable_pairs = json.load(f)

# Retrieve the candidate ID
candidates_by_id = {c["candidate_id"]: c for c in candidates}

In [131]:
# Define a function to verify and edit QA pairs
def verify_and_edit(candidate_id, new_question = None, new_answer =None):
    idx = next(i for i, p in enumerate(answerable_pairs) if p["candidate_id"] == candidate_id)
    answerable_pairs[idx]["human_verified"] = True
    if new_question:
        answerable_pairs[idx]["question"] = new_question
    if new_answer:
        answerable_pairs[idx]["answer"] = new_answer
    return answerable_pairs[idx]

In [434]:
# Load the exchange corpus to view the source documents
with open("unified_text_corpus.json", "r", encoding = "utf-8") as f:
    corpus = json.load(f)
corpus_by_id = {d["id"]: d for d in corpus}

# Define a function to show the QAs with their source docs for review
def review_qa(pair):
    print(f"=== {pair['candidate_id']} ({pair['category']}, {pair['exchange']}) ===")
    print(f"\nQUESTION: {pair['question']}")
    print(f"ANSWER: {pair['answer']}")
    print("\n--- Source excerpt(s) (looked up by stable source_doc_id, not candidate_id) ---")

    if pair["category"] == "comparative":
        doc_a = corpus_by_id.get(pair.get("source_doc_id_earlier"))
        doc_b = corpus_by_id.get(pair.get("source_doc_id_later"))
        if doc_a:
            print(f"\n[{doc_a.get('date')}] {doc_a['title']}\n{doc_a['text'][:1500]}")
        if doc_b:
            print(f"\n[{doc_b.get('date')}] {doc_b['title']}\n{doc_b['text'][:1500]}")
    else:
        doc = corpus_by_id.get(pair.get("source_doc_id"))
        if doc:
            print(f"\n[{doc.get('date')}] {doc['title']}\n{doc['text'][:1500]}")
        else:
            print("Could not find source document by source_doc_id.")
    print("\n" + "="*80)

# Print the number of unverified pairs
unverified_pairs = [p for p in answerable_pairs if not p.get("human_verified")]
print(f"{len(unverified_pairs)} remaining to review")
    
# Review one at a time, change the position and re-run
review_position = 0
review_qa(unverified_pairs[review_position])

55 remaining to review
=== jse-factual-4 (factual, jse) ===

QUESTION: What is the name of the indices provided by the company on August 22, 2025?
ANSWER: JSE Bond Indices

--- Source excerpt(s) (looked up by stable source_doc_id, not candidate_id) ---

[2025-08-22T17:06:32] JSE Bond Indices as at August 22, 2025
JSE Bond Indices August 22, 2025



In [437]:
# Verify and edit the question or answer if needed
verify_and_edit("jse-factual-4"
    ,new_question = "What is the company name of the JSE trading symbol 138SL?"
    ,new_answer = "138 STUDENT LIVING JAMAICA LIMITED")

# Save the update to the QA pair
with open("qa_draft_set.json", "w", encoding="utf-8") as f:
    json.dump(answerable_pairs, f, indent = 2, ensure_ascii = False)

# Print the number of unverified pairs after the update
verified_count = sum(1 for p in answerable_pairs if p["human_verified"])
print(f"Pairs Verified: {verified_count} / {len(answerable_pairs)}")

Pairs Verified: 138 / 192


In [393]:
# To view the complete source document, run this code

# Replace with candidate_id that being verified
candidate = candidates_by_id["bse-event-63"]

# Find the source and target doucment by source ID
target_doc_id = candidate.get("source_doc_id")
full_doc = corpus_by_id.get(target_doc_id)

# Show the full document if it exists
if full_doc:
    print(f"Title: {full_doc['title']}")
    print(f"Date: {full_doc['date']}")
    print(full_doc["text"])
else:
    print(f"Couldn't find a document with id={target_doc_id!r}")

Title: Cancellation of Shares – West India Biscuit Company Limited
Date: 2025-06-24T14:39:38
West India Biscuit Company Limited – Please be advised that a total of 1,013 shares in West India Biscuit Company Limited were re-purchased by the company and cancelled by the Barbados Stock Exchange Inc.


In [444]:
# Generate the final verified numbers by exchange and category
    
# Rebuild fresh from the live in-memory list -- not a stale df from earlier in the session
df_qa = pd.DataFrame(answerable_pairs)

# Verified vs. total by exchange x category
summary = df_qa.groupby(["exchange", "category"]).agg(
    verified = ("human_verified", "sum"),
    total = ("human_verified", "count"),
).reset_index()
summary["verified / total"] = summary["verified"].astype(str) + " / " + summary["total"].astype(str)

pivot = summary.pivot(index = "exchange", columns = "category", values = "verified / total")
print("### Verified by Exchange x Category\n")
print(pivot.to_markdown())

# Number by questions by type and exchanges
if "comparative_type" in qa_df.columns:
    comparative_df = qa_df[qa_df["category"] == "comparative"]
    comp_summary = comparative_df.groupby(["exchange", "comparative_type"]).agg(
        verified = ("human_verified", "sum"),
        total = ("human_verified", "count"),).reset_index()
    comp_summary["verified / total"] = comp_summary["verified"].astype(str) + " / " + comp_summary["total"].astype(str)
    comp_pivot = comp_summary.pivot(index="exchange", columns="comparative_type", values="verified / total")
    print("\n### Comparative Verified by Type\n")
    print(comp_pivot.to_markdown())

# Print the total number of verified questions
print(f"\n### Overall: {qa_df['human_verified'].sum()} / {len(qa_df)} verified")

### Verified by Exchange x Category

| exchange   | comparative   | event   | factual   |
|:-----------|:--------------|:--------|:----------|
| bse        | 12 / 22       | 17 / 22 | 17 / 21   |
| jse        | 12 / 22       | 17 / 17 | 17 / 17   |
| ttse       | 12 / 21       | 17 / 27 | 17 / 23   |

### Overall: 138 / 192 verified
